# 11 — Restitution décisionnelle

> **Public visé : décideur non statisticien.**  
> Cette restitution utilise exclusivement les résultats calculés dans les notebooks 05 à 10.
> Les données du projet sont simulées et reproductibles : les valeurs illustrent une démarche
> méthodologique et ne constituent pas une estimation officielle de la DEPP.


In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if (Path.cwd() / "notebooks").exists() is False else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd

from src.data import load_processed_dataset
from src.utils.paths import TABLES_DIR

df = load_processed_dataset("education_prioritaire")
indicateurs = pd.read_csv(TABLES_DIR / "indicateurs_decisionnels.csv")
comparaison_statut = pd.read_csv(TABLES_DIR / "05_comparaison_statut.csv")
tendances = pd.read_csv(TABLES_DIR / "08_tendance_lineaire_par_statut.csv")
ecarts_temps = pd.read_csv(TABLES_DIR / "08_ecart_repplus_vs_horsep_temps.csv")
faisabilite_causale = pd.read_csv(TABLES_DIR / "09_faisabilite_causale_did.csv")
clusters = pd.read_csv(TABLES_DIR / "07_clusters_profils_ecoles.csv")

assert df.shape == (5250, 21)
assert len(indicateurs) == 9
print(f"Données analysées : {df.shape[0]:,} observations, {df['ecole_id'].nunique()} écoles, "
      f"{df['annee'].nunique()} années et {df['niveau'].nunique()} niveaux.")


Données analysées : 5,250 observations, 150 écoles, 7 années et 5 niveaux.


## 1. Question posée

**Question de pilotage.** Les écarts de performance scolaire entre les établissements relevant de
l'éducation prioritaire (REP/REP+) et ceux hors éducation prioritaire évoluent-ils au cours du temps,
et quelles informations disponibles permettent de mieux cibler le suivi ?

**Important.** Cette restitution décrit des écarts et des tendances observés. Elle **ne mesure pas**
l'effet causal du dédoublement des classes : les conditions nécessaires ne sont pas réunies dans les
données actuelles (voir section 9).


## 2. Données utilisées

- **5 250 observations** de scores globaux (échelle 0–100) ;
- **150 écoles**, suivies de **2017 à 2023** ;
- niveaux : CP, CE1, CM1, CM2 et 6e ;
- variables utilisées : statut (Hors EP, REP, REP+), IPS, taille moyenne de classe, exposition au
  dédoublement et effectifs ;
- source : jeu de données **simulé, reproductible**, construit pour reproduire la structure du cas d'étude.

Les données sont donc suffisantes pour une lecture descriptive, des tendances agrégées et une
segmentation des écoles ; elles ne sont pas une base administrative individuelle.


## 3. Méthode

1. Comparaison descriptive des scores par statut.
2. Suivi annuel des moyennes 2017–2023.
3. Modèle explicatif associatif : statut, niveau, IPS et dédoublement ; erreurs groupées par école.
4. Segmentation de 150 écoles en trois profils opérationnels.
5. Test de faisabilité causal (différence-de-différences) : **non concluant / non applicable**.

Les graphiques détaillés sont disponibles dans `outputs/figures/05_*`, `07_*` et `08_*`.


## 4. Cinq résultats majeurs

In [2]:
moyennes = comparaison_statut.set_index("statut")["moyenne"]
trend = tendances.set_index("statut")
gap_2017 = ecarts_temps.loc[ecarts_temps["annee"] == 2017, "ecart_compare_moins_ref"].iloc[0]
gap_2023 = ecarts_temps.loc[ecarts_temps["annee"] == 2023, "ecart_compare_moins_ref"].iloc[0]
modele_r2 = indicateurs.loc[indicateurs["nom"] == "R² modèle explicatif OLS", "valeur"].iloc[0]
rmse = indicateurs.loc[indicateurs["nom"] == "RMSE modèle explicatif OLS", "valeur"].iloc[0]
couverture = indicateurs.loc[indicateurs["nom"] == "Part d'observations dédoublées", "valeur"].iloc[0]

resultats_majeurs = pd.DataFrame(
    [
        ["1. Écart de niveau", f"REP+ : {moyennes['REP+']:.1f} ; Hors EP : {moyennes['Hors EP']:.1f}. Écart : {moyennes['REP+'] - moyennes['Hors EP']:.1f} points.",
         "Les performances moyennes observées restent plus basses en REP+.", "Cibler le suivi des inégalités sur REP+.", "Suivre cet écart chaque année, sans l'attribuer au seul dédoublement."],
        ["2. Progression temporelle", f"Pente REP+ : +{trend.loc['REP+', 'pente_par_an']:.2f} point/an ; Hors EP : +{trend.loc['Hors EP', 'pente_par_an']:.2f} point/an.",
         "Les deux groupes progressent sur la période.", "Une amélioration globale est observable.", "Maintenir un suivi des tendances séparé par statut."],
        ["3. Écart persistant", f"Écart REP+ - Hors EP : {gap_2017:.1f} points en 2017 ; {gap_2023:.1f} points en 2023.",
         "L'écart reste négatif tout au long de la période et ne se résorbe pas.", "La progression moyenne ne suffit pas à réduire les inégalités.", "Ajouter un objectif explicite de réduction des écarts, pas seulement de hausse moyenne."],
        ["4. Variables explicatives", f"Modèle associatif : R² = {modele_r2:.3f} ; erreur moyenne quadratique = {rmse:.1f} points.",
         "Statut, niveau et IPS sont associés à une large part des différences de scores.", "Le contexte social et scolaire doit être intégré au pilotage.", "Comparer les établissements à contexte comparable (notamment IPS)."],
        ["5. Couverture du dispositif", f"{couverture:.1%} des observations sont marquées comme dédoublées dans ce panel.",
         "L'exposition observée est partielle et ne permet pas d'attribuer causalement un effet.", "La couverture seule ne renseigne pas sur l'efficacité.", "Documenter précisément le dispositif réel, niveau et année d'application."],
    ],
    columns=["resultat", "resultat_statistique", "interpretation", "implication_metier", "recommandation"],
)
resultats_majeurs


,resultat,resultat_statistique,interpretation,implication_metier,recommandation
0,1. Écart de niveau,REP+ : 71.5 ; Hors EP : 81.7. Écart : -10.2 po...,Les performances moyennes observées restent pl...,Cibler le suivi des inégalités sur REP+.,"Suivre cet écart chaque année, sans l'attribue..."
1,2. Progression temporelle,Pente REP+ : +0.84 point/an ; Hors EP : +0.81 ...,Les deux groupes progressent sur la période.,Une amélioration globale est observable.,Maintenir un suivi des tendances séparé par st...
2,3. Écart persistant,Écart REP+ - Hors EP : -8.8 points en 2017 ; -...,L'écart reste négatif tout au long de la pério...,La progression moyenne ne suffit pas à réduire...,Ajouter un objectif explicite de réduction des...
3,4. Variables explicatives,Modèle associatif : R² = 0.703 ; erreur moyenn...,"Statut, niveau et IPS sont associés à une larg...",Le contexte social et scolaire doit être intég...,Comparer les établissements à contexte compara...
4,5. Couverture du dispositif,33.5% des observations sont marquées comme déd...,L'exposition observée est partielle et ne perm...,La couverture seule ne renseigne pas sur l'eff...,"Documenter précisément le dispositif réel, niv..."


### Séparation de lecture

Pour chaque ligne du tableau ci-dessus :

**Résultat statistique** → valeur calculée dans les analyses précédentes.  
**Interprétation** → ce que cette valeur décrit.  
**Implication métier** → ce que le décideur peut en retenir.  
**Recommandation** → action prudente et concrète, sans surinterprétation causale.


## 5. Populations et groupes les plus concernés

In [3]:
groupes = comparaison_statut[["statut", "effectif", "moyenne", "ic_95_basse", "ic_95_haute"]].copy()
groupes["moyenne"] = groupes["moyenne"].round(2)
groupes


,statut,effectif,moyenne,ic_95_basse,ic_95_haute
0,Hors EP,2940,81.71,81.335188,82.079077
1,REP,1460,75.17,74.715804,75.633580
2,REP+,850,71.46,70.891331,72.024646


In [4]:
clusters.rename(
    columns={
        "score_moyen": "score moyen",
        "ips_moyen": "IPS moyen",
        "taille_moyenne_classe": "taille moyenne de classe",
        "part_dedoublement": "part dédoublée",
        "effectif_ecoles": "nombre d'écoles",
    }
)


,cluster,score moyen,IPS moyen,taille moyenne de classe,part dédoublée,nombre d'écoles
0,0,79.963974,73.248546,16.820893,0.214286,56
1,2,77.739429,72.597774,15.501096,0.538206,43
2,1,76.742902,72.488964,18.056471,0.296919,51


**Résultat statistique.** REP+ est le groupe au score moyen le plus faible ; le clustering identifie
un profil de 43 écoles avec la part d'exposition au dédoublement la plus élevée (~53,8 %) et une taille
moyenne de classe la plus basse (~15,5 élèves).

**Interprétation.** Les besoins ne sont pas homogènes : le statut et le profil d'école doivent être lus
conjointement.

**Implication métier.** Une politique uniforme risque de manquer les contrastes entre établissements.

**Recommandation.** Organiser le suivi par statut et profil d'école, puis investiguer les écoles dont la
trajectoire s'écarte fortement de leur profil.


## 6. Points de vigilance

- L'écart observé est une **association**, pas une mesure de l'effet du dédoublement.
- Les scores sont issus de données simulées/reconstruites : aucune valeur ne doit être présentée comme
  un résultat officiel de la DEPP.
- Les analyses sont agrégées au niveau école ; aucune trajectoire élève n'est disponible.
- Les statuts d'école changent fréquemment dans le jeu actuel, limitant l'interprétation institutionnelle.
- Les indicateurs de modèle sont calculés sur l'échantillon observé, sans validation externe.


## 7. Implications

**Résultat statistique.** L'écart REP+ – Hors EP demeure proche de 10 points en 2023, malgré une
progression des scores moyens dans les deux groupes.

**Interprétation.** Une amélioration globale du niveau ne garantit pas la réduction des inégalités.

**Implication métier.** Le pilotage devrait suivre simultanément :
1. le niveau moyen ;
2. l'écart entre statuts ;
3. la progression annuelle ;
4. le profil de contexte des écoles (IPS, taille, exposition).

**Recommandation.** Mettre ces quatre dimensions dans un tableau de bord annuel, avec un indicateur
d'alerte lorsque l'écart REP+ – Hors EP s'accroît.


## 8. Recommandations

| Priorité | Recommandation | Justification issue des résultats | Prudence |
|---|---|---|---|
| 1 | Suivre annuellement l'écart REP+ / Hors EP | Écart moyen observé ≈ -10,25 points ; persistant de 2017 à 2023. | Ne pas l'interpréter comme un effet causal. |
| 2 | Comparer à IPS et niveau comparables | IPS et statut sont associés aux scores dans le modèle. | Ajouter les covariables manquantes avec des données réelles. |
| 3 | Segmenter le pilotage des écoles | Trois profils d'écoles émergent du clustering. | Profils exploratoires, à valider sur données administratives. |
| 4 | Documenter l'exposition réelle au dédoublement | La variable actuelle ne respecte pas le déploiement CP/CE1 attendu. | Prérequis avant toute évaluation causale. |
| 5 | Construire une base réelle pseudo-cohorte | Analyse longitudinale faisable à l'école, pas à l'élève. | Définir clés de jointure, qualité et confidentialité. |


## 9. Limites

### Limite principale : causalité non identifiable
Le test de faisabilité conclut **NON APPLICABLE** pour une différence-de-différences robuste :
- 100 % des écoles changent de statut au moins une fois dans la période ;
- la part d'exposition au dédoublement est identique en CP/CE1 et dans les autres niveaux dans le jeu
  actuel (écart d'alignement = 0).

En conséquence :
- **corrélation** : oui, elle décrit une co-variation ;
- **association** : oui, le modèle contrôle certaines variables ;
- **prédiction** : oui, le modèle produit une estimation de score ;
- **causalité** : non, aucune preuve causale n'est revendiquée.


## 10. Travaux complémentaires

1. Remplacer la simulation par les fichiers Open Data consolidés et documenter les sources/versions.
2. Reconstruire des pseudo-cohortes avec des identifiants école et des règles de jointure stables.
3. Enregistrer le calendrier réel d'exposition CP/CE1 (REP+ 2017, élargissements ultérieurs).
4. Tester les tendances pré-traitement et constituer un groupe de comparaison stable avant toute DiD.
5. Réaliser une validation temporelle/externe du modèle prédictif.
6. Compléter par des variables de contexte disponibles : composition sociale, caractéristiques territoriales,
   stabilité des équipes et moyens alloués.

Ces prérequis permettraient de passer d'une restitution descriptive fiable à une évaluation de politique
publique causalement défendable.
